In [12]:
# Install only when Kaggle's current environment does not already include Ultralytics.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])


In [13]:
from pathlib import Path
import os, json, random, cv2
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score, mean_absolute_error
)
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT = Path("/kaggle/input")
RAM_ROOT = Path("/kaggle/input/datasets/thujainanajumuddin/ram-h1200-research")
WORK = Path("/kaggle/working/ra_project")
CHECKPOINTS = WORK / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("RAM-H1200 exists:", RAM_ROOT.is_dir())
print("Output folder:", WORK)

Device: cuda
RAM-H1200 exists: True
Output folder: /kaggle/working/ra_project


In [14]:
# Locate input datasets without recursively walking every image directory.
# Kaggle normally mounts datasets as /kaggle/input/datasets/<owner>/<slug>/...
ra_candidates = [
    *INPUT.glob("datasets/*/*/dataset"),
    *INPUT.glob("*/dataset"),
]
RA_ROOT = next((p for p in ra_candidates if all(
    (p / split / "Normal").is_dir() and
    any((p / split / grade).is_dir() for grade in ("Mild", "Moderate", "Severe"))
    for split in ("train", "val", "test")
)), None)
if RA_ROOT is None:
    raise FileNotFoundError(
        "RA image dataset not found. Attach the dataset that contains "
        "dataset/{train,val,test}/{Normal,Mild,Moderate,Severe}."
    )

mura_candidates = [
    INPUT / "datasets/cjinny/mura-v11/MURA-v1.1",
    INPUT / "cjinny/mura-v11/MURA-v1.1",
    *INPUT.glob("datasets/*/*/MURA-v1.1"),
    *INPUT.glob("*/MURA-v1.1"),
]
MURA_ROOT = next((p for p in mura_candidates if
                  (p / "train" / "XR_HAND").is_dir() and
                  (p / "valid" / "XR_HAND").is_dir()), None)
if MURA_ROOT is None:
    raise FileNotFoundError(
        "MURA-v1.1 not found. Attach MURA v1.1 and confirm it contains "
        "train/XR_HAND and valid/XR_HAND."
    )

print("RA image dataset:", RA_ROOT)
print("MURA root:", MURA_ROOT)
print("RAM-H1200 root:", RAM_ROOT, "exists:", RAM_ROOT.is_dir())


RA image dataset: /kaggle/input/datasets/sindhuvunna/rheumatoid-arthritis-detection/dataset
MURA root: /kaggle/input/datasets/cjinny/mura-v11/MURA-v1.1
RAM-H1200 root: /kaggle/input/datasets/thujainanajumuddin/ram-h1200-research exists: True


In [15]:
assert RAM_ROOT.is_dir(), f"RAM-H1200 not found: {RAM_ROOT}"
assert MURA_ROOT.is_dir(), f"MURA not found: {MURA_ROOT}"

for relative in [
    "Segmentation/train",
    "Segmentation/val",
    "Segmentation/test",
    "SvdH_Scoring/SvdH_JSN_Scoring/train",
    "SvdH_Scoring/SvdH_BE_Scoring/train",
]:
    folder = RAM_ROOT / relative
    print(relative, "exists:", folder.is_dir())


Segmentation/train exists: True
Segmentation/val exists: True
Segmentation/test exists: True
SvdH_Scoring/SvdH_JSN_Scoring/train exists: True
SvdH_Scoring/SvdH_BE_Scoring/train exists: True


In [16]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MURA_PARTS = ("XR_HAND", "XR_WRIST", "XR_FINGER")


def image_tensor(path, augment=False):
    """DIP preprocessing: grayscale decode, CLAHE, 224x224 resize, tensor scaling."""
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"Could not read image: {path}")
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    gray = cv2.resize(gray, (224, 224), interpolation=cv2.INTER_AREA)
    if augment and random.random() < 0.5:
        gray = cv2.flip(gray, 1)
    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB).astype(np.float32) / 255.0
    rgb = (rgb - 0.5) / 0.5
    return torch.from_numpy(rgb.transpose(2, 0, 1).copy())


def collect_mura_negative(split):
    """Collect MURA normal studies only for hand, wrist, and finger."""
    root = MURA_ROOT / split
    paths = []
    for part in MURA_PARTS:
        part_root = root / part
        if not part_root.is_dir():
            continue
        paths.extend(
            p for p in part_root.rglob("*.png")
            if "study" in p.parent.name.lower()
            and "negative" in p.parent.name.lower()
            and not p.name.startswith("._")
        )
    return sorted(paths)


# Keep all MURA training patients in training. Split the official MURA validation
# patients into separate validation and test controls so no patient is shared.
mura_train_paths = collect_mura_negative("train")
mura_valid_paths = collect_mura_negative("valid")
if not mura_train_paths or len(mura_valid_paths) < 2:
    raise ValueError("Could not find enough MURA negative hand/wrist/finger images.")

mura_valid_patients = sorted({p.parent.parent.name for p in mura_valid_paths})
random.Random(SEED).shuffle(mura_valid_patients)
cut = max(1, min(len(mura_valid_patients) - 1, len(mura_valid_patients) // 2))
val_patients = set(mura_valid_patients[:cut])
test_patients = set(mura_valid_patients[cut:])
mura_normal_by_split = {
    "train": mura_train_paths,
    "val": [p for p in mura_valid_paths if p.parent.parent.name in val_patients],
    "test": [p for p in mura_valid_paths if p.parent.parent.name in test_patients],
}

class RADataset(Dataset):
    """Binary data: MURA negative studies=Normal; RA Mild/Moderate/Severe=RA."""
    def __init__(self, split, augment=False):
        self.samples = [(p, 0) for p in mura_normal_by_split[split]]
        self.augment = augment
        for class_name in ("Mild", "Moderate", "Severe"):
            folder = RA_ROOT / split / class_name
            if folder.is_dir():
                self.samples.extend(
                    (p, 1) for p in folder.rglob("*")
                    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
                    and not p.name.startswith("._")
                )
        if not self.samples:
            raise ValueError(f"No binary images assembled for {split}.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        return image_tensor(path, self.augment), torch.tensor(label, dtype=torch.float32)


In [17]:
train_ds = RADataset("train", augment=True)
val_ds = RADataset("val")
test_ds = RADataset("test")

# num_workers=0 avoids Kaggle/Python 3.13 worker-shutdown warnings seen in this notebook.
LOADER_WORKERS = 0
train_loader = DataLoader(
    train_ds, batch_size=16, shuffle=True, num_workers=LOADER_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=16, shuffle=False, num_workers=LOADER_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=16, shuffle=False, num_workers=LOADER_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

counts = {label: sum(y == label for _, y in train_ds.samples) for label in (0, 1)}
if min(counts.values()) == 0:
    raise ValueError(f"Training split must contain both classes; counts={counts}")
BINARY_POS_WEIGHT = torch.tensor(counts[0] / counts[1], dtype=torch.float32, device=DEVICE)
print("Train:", len(train_ds), "| validation:", len(val_ds), "| test:", len(test_ds))
print("Binary train counts (Normal, RA):", counts)
print("MURA validation patients:", len(val_patients), "| MURA test patients:", len(test_patients))


Train: 14958 | validation: 845 | test: 862
Binary train counts (Normal, RA): {0: 12962, 1: 1996}
MURA validation patients: 138 | MURA test patients: 138


In [18]:
from collections import Counter

for name, dataset in [
    ("train", train_ds),
    ("validation", val_ds),
    ("test", test_ds),
]:
    print(name, Counter(label for _, label in dataset.samples))

train Counter({0: 12962, 1: 1996})
validation Counter({1: 426, 0: 419})
test Counter({1: 432, 0: 430})


In [19]:
class DenseNetBinaryClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=0.25):
        super().__init__()
        weights = DenseNet121_Weights.DEFAULT if pretrained else None
        self.backbone = densenet121(weights=weights)
        features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(features, 1)

    def forward(self, x):
        return self.classifier(self.dropout(self.backbone(x))).squeeze(1)

binary_model = DenseNetBinaryClassifier(pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(binary_model.parameters(), lr=1e-4, weight_decay=1e-4)

In [20]:
def binary_pass(loader, training=False):
    """One pass returning mean BCE loss and sample-level predictions."""
    binary_model.train(training)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=BINARY_POS_WEIGHT)
    loss_sum, count = 0.0, 0
    actual, predicted = [], []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = binary_model(images)
            loss = loss_fn(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        pred = (logits.detach() >= 0).long()
        loss_sum += loss.item() * len(labels)
        count += len(labels)
        actual.extend(labels.long().cpu().tolist())
        predicted.extend(pred.cpu().tolist())
    return loss_sum / max(count, 1), actual, predicted


In [21]:
binary_path = CHECKPOINTS / "ra_binary.pt"
best_val_loss = float("inf")
BINARY_EPOCHS = 10

for epoch in range(1, BINARY_EPOCHS + 1):
    train_loss, train_true, train_pred = binary_pass(train_loader, training=True)
    val_loss, val_true, val_pred = binary_pass(val_loader)
    train_acc = accuracy_score(train_true, train_pred)
    val_acc = accuracy_score(val_true, val_pred)
    print(
        f"Epoch {epoch}/{BINARY_EPOCHS} | "
        f"train loss {train_loss:.4f} | train accuracy {train_acc:.3f} | "
        f"val loss {val_loss:.4f} | val accuracy {val_acc:.3f}"
    )
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"state_dict": binary_model.state_dict(),
                    "image_size": 224, "epoch": epoch,
                    "best_val_loss": best_val_loss}, binary_path)
        print("  Saved best validation checkpoint:", binary_path)

# Evaluate the best checkpoint once on train and once on the untouched test split.
saved = torch.load(binary_path, map_location=DEVICE, weights_only=False)
binary_model.load_state_dict(saved["state_dict"])
train_eval_loss, train_eval_true, train_eval_pred = binary_pass(train_loader)
test_loss, test_true, test_pred = binary_pass(test_loader)
print("\nBest checkpoint epoch:", saved["epoch"])
print(f"Final train loss: {train_eval_loss:.4f} | training accuracy: {accuracy_score(train_eval_true, train_eval_pred):.3f}")
print("Training confusion matrix (rows=true, columns=predicted; Normal, RA):")
print(confusion_matrix(train_eval_true, train_eval_pred, labels=[0, 1]))
print(f"Held-out test loss: {test_loss:.4f} | test accuracy: {accuracy_score(test_true, test_pred):.3f}")
print("Test confusion matrix (rows=true, columns=predicted; Normal, RA):")
print(confusion_matrix(test_true, test_pred, labels=[0, 1]))
print(classification_report(
    test_true, test_pred, labels=[0, 1],
    target_names=["Normal", "RA"], zero_division=0
))
print("Checkpoint:", binary_path)


  0%|          | 0/935 [00:00<?, ?it/s]

  0%|          | 0/53 [00:00<?, ?it/s]

Epoch 1/10 | train loss 0.0423 | train accuracy 0.993 | val loss 0.0146 | val accuracy 1.000
  Saved best validation checkpoint: /kaggle/working/ra_project/checkpoints/ra_binary.pt


  0%|          | 0/935 [00:00<?, ?it/s]

  0%|          | 0/53 [00:00<?, ?it/s]

...

  0%|          | 0/53 [00:00<?, ?it/s]

Epoch 10/10 | train loss 0.0002 | train accuracy 1.000 | val loss 0.0001 | val accuracy 1.000


  0%|          | 0/935 [00:00<?, ?it/s]

  0%|          | 0/54 [00:00<?, ?it/s]


Best checkpoint epoch: 9
Final train loss: 0.0000 | training accuracy: 1.000
Training confusion matrix (rows=true, columns=predicted; Normal, RA):
[[12962     0]
 [    0  1996]]
Held-out test loss: 0.0000 | test accuracy: 1.000
Test confusion matrix (rows=true, columns=predicted; Normal, RA):
[[430   0]
 [  0 432]]
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       430
          RA       1.00      1.00      1.00       432

    accuracy                           1.00       862
   macro avg       1.00      1.00      1.00       862
weighted avg       1.00      1.00      1.00       862

Checkpoint: /kaggle/working/ra_project/checkpoints/ra_binary.pt


In [22]:
from ultralytics import YOLO

YOLO_ROOT = WORK / "ram_h1200_yolo"
for split in ("train", "val", "test"):
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

print("YOLO output folder:", YOLO_ROOT)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO output folder: /kaggle/working/ra_project/ram_h1200_yolo


In [23]:
def find_annotation(folder, filename):
    matches = list(folder.glob(f"**/{filename}"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {filename} below {folder}; found {len(matches)}"
        )
    return matches[0]

def prepare_yolo_split(split):
    task_root = RAM_ROOT / "SvdH_Scoring" / "SvdH_JSN_Scoring" / split
    ann_path = find_annotation(task_root, "_annotations_jsn_joint_detection.coco.json")
    coco = json.loads(ann_path.read_text())

    # COCO may include an unused parent category; expose only categories with boxes.
    used_ids = {item["category_id"] for item in coco["annotations"]}
    categories = sorted(
        (item for item in coco["categories"] if item["id"] in used_ids),
        key=lambda item: item["id"],
    )
    category_to_class = {item["id"]: i for i, item in enumerate(categories)}
    class_names = [item["name"] for item in categories]

    source_root = RAM_ROOT / "Segmentation" / split
    source_images = {
        p.name: p for p in source_root.iterdir()
        if p.is_file() and p.suffix.lower() == ".bmp"
    }
    image_records = {item["id"]: item for item in coco["images"]}
    annotations = {}
    for item in coco["annotations"]:
        annotations.setdefault(item["image_id"], []).append(item)

    for image_id, record in image_records.items():
        source = source_images.get(Path(record["file_name"]).name)
        if source is None:
            raise FileNotFoundError(f"Annotated image missing: {record['file_name']}")
        dest_image = YOLO_ROOT / "images" / split / source.name
        if not dest_image.exists():
            os.symlink(source, dest_image)
        width, height = record["width"], record["height"]
        lines = []
        for box in annotations.get(image_id, []):
            x, y, w, h = box["bbox"]
            if w <= 0 or h <= 0 or box["category_id"] not in category_to_class:
                continue
            cx, cy = (x + w / 2) / width, (y + h / 2) / height
            lines.append(
                f"{category_to_class[box['category_id']]} "
                f"{cx:.6f} {cy:.6f} {w/width:.6f} {h/height:.6f}"
            )
        (YOLO_ROOT / "labels" / split / f"{source.stem}.txt").write_text("\n".join(lines))
    return class_names, len(image_records), len(coco["annotations"])


In [24]:
class_names = None

for split in ("train", "val", "test"):
    names, image_count, box_count = prepare_yolo_split(split)
    class_names = names
    print(f"{split}: {image_count} images, {box_count} boxes")

yaml_lines = [
    f"path: {YOLO_ROOT}",
    "train: images/train",
    "val: images/val",
    "test: images/test",
    "names:",
]
yaml_lines += [f"  {i}: '{name}'" for i, name in enumerate(class_names)]

yaml_path = YOLO_ROOT / "data.yaml"
yaml_path.write_text("\n".join(yaml_lines) + "\n")
print("YOLO data config:", yaml_path)
print("Classes:", class_names)

train: 793 images, 11895 boxes
val: 140 images, 2100 boxes
test: 267 images, 4005 boxes
YOLO data config: /kaggle/working/ra_project/ram_h1200_yolo/data.yaml
Classes: ['CMC-M', 'CMC-R', 'CMC-S', 'SC', 'SR', 'STT', 'MCP-T', 'MCP-I', 'MCP-M', 'MCP-R', 'MCP-S', 'PIP-I', 'PIP-M', 'PIP-R', 'PIP-S']


In [25]:
yolo_model = YOLO("yolov8n.pt")
yolo_model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=8,
    device=0 if DEVICE.type == "cuda" else "cpu",
    workers=2,
    patience=8,
    project=str(WORK / "yolo_runs"),
    name="ram_joint_detector",
    save=True,
)

YOLO_BEST = WORK / "yolo_runs" / "ram_joint_detector" / "weights" / "best.pt"
print("Best detector checkpoint:", YOLO_BEST, "| exists:", YOLO_BEST.is_file())

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ra_project/ram_h1200_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ram_joint_detector, nbs=64, nms=None, 

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      2.56G       1.35      1.915        1.1         18        640: 100% ━━━━━━━━━━━━ 100/100 13.4it/s 7.5s.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 12.0it/s 0.8s.2s
                   all        140       2100      0.722      0.644      0.705      0.473

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      2.56G      1.129       1.17      1.015         23        640: 100% ━━━━━━━━━━━━ 100/100 13.4it/s 7.5s.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 12.2it/s 0.7s.1s
                   all        140       2100      0.826      0.788      0.883       0.61

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      2.56G      1.067      0.953     0.9895         36        640: 100% ━━━━━━━━━━━━ 100/100 13.4it/s 7.5s.1s
                 Class     Images  Insta

In [26]:
# Separate cell: evaluate using the held-out test split.
YOLO(str(YOLO_BEST)).val(
    data=str(yaml_path),
    split="test",
    imgsz=640,
    device=0 if DEVICE.type == "cuda" else "cpu",
)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 96.0±22.2 MB/s, size: 1588.8 KB)
val: Scanning /kaggle/working/ra_project/ram_h1200_yolo/labels/test... 267 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 267/267 273.1it/s 1.0s0.1s
val: New cache created: /kaggle/working/ra_project/ram_h1200_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 4.0it/s 4.3s0.1s
                   all        267       4005      0.996      0.995      0.994      0.793
                 CMC-M        267        267      0.998      0.993      0.995       0.63
                 CMC-R        267        267      0.989      0.987      0.994      0.611
                 CMC-S        267        267          1      0.998      0.995      0.671
                    SC      

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78ffec6232a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,   

In [27]:
def load_score_dictionary(path):
    data = json.loads(path.read_text())
    for wrapper in ("scores", "annotations", "data"):
        if isinstance(data, dict) and isinstance(data.get(wrapper), dict):
            data = data[wrapper]
            break
    if not isinstance(data, dict):
        raise ValueError(f"Unexpected score JSON structure: {path}")
    return data

def build_score_rows(task, prefix, split):
    task_root = RAM_ROOT / "SvdH_Scoring" / f"SvdH_{task}_Scoring" / split
    score_filename = f"_annotation_{prefix.lower()}_scores.json"
    score_path = find_annotation(task_root, score_filename)
    scores = load_score_dictionary(score_path)
    rows = []

    for case_filename, joint_scores in scores.items():
        if not isinstance(joint_scores, dict):
            continue
        case_folder = task_root / Path(case_filename).stem

        for key, grade in joint_scores.items():
            if not str(key).startswith(prefix + "_"):
                continue
            joint = str(key)[len(prefix) + 1:]
            crop = next(
                (case_folder / f"{joint}{ext}"
                 for ext in (".bmp", ".png", ".jpg")
                 if (case_folder / f"{joint}{ext}").is_file()),
                None
            )
            if crop is not None:
                rows.append((crop, int(grade)))

    if not rows:
        raise ValueError(f"No {prefix} scored crops matched in {task_root}")
    return rows

In [28]:
severity_rows = {
    "JSN": {
        split: build_score_rows("JSN", "JSN", split)
        for split in ("train", "val", "test")
    },
    "BE": {
        split: build_score_rows("BE", "BE", split)
        for split in ("train", "val", "test")
    },
}

for task in ("JSN", "BE"):
    print(task, {split: len(rows) for split, rows in severity_rows[task].items()})

JSN {'train': 11895, 'val': 2100, 'test': 4005}
BE {'train': 12688, 'val': 2240, 'test': 4272}


In [29]:
class SeverityDataset(Dataset):
    def __init__(self, rows, augment=False):
        self.rows = rows
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        path, grade = self.rows[index]
        return image_tensor(path, self.augment), torch.tensor(grade, dtype=torch.long)

class OrdinalHead(nn.Module):
    def __init__(self, features, grades):
        super().__init__()
        self.score = nn.Linear(features, 1)
        self.first_threshold = nn.Parameter(torch.tensor(0.0))
        self.threshold_deltas = nn.Parameter(torch.zeros(grades - 2))

    def forward(self, features):
        if self.threshold_deltas.numel():
            increments = torch.nn.functional.softplus(self.threshold_deltas) + 1e-4
            thresholds = torch.cat([
                self.first_threshold.reshape(1),
                self.first_threshold + torch.cumsum(increments, dim=0)
            ])
        else:
            thresholds = self.first_threshold.reshape(1)
        return self.score(features) - thresholds.unsqueeze(0)

In [30]:
class OrdinalSeverityClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=0.25):
        super().__init__()
        weights = DenseNet121_Weights.DEFAULT if pretrained else None
        self.backbone = densenet121(weights=weights)
        features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.jsn_head = OrdinalHead(features, grades=5)
        self.erosion_head = OrdinalHead(features, grades=6)

    def forward(self, x):
        features = self.dropout(self.backbone(x))
        return {
            "jsn_logits": self.jsn_head(features),
            "erosion_logits": self.erosion_head(features),
        }

severity_model = OrdinalSeverityClassifier(pretrained=True).to(DEVICE)
severity_optimizer = torch.optim.AdamW(
    severity_model.parameters(), lr=1e-4, weight_decay=1e-4
)

In [31]:
def ordinal_targets(grades, thresholds):
    values = torch.arange(thresholds, device=grades.device)
    return (grades[:, None] > values[None, :]).float()

def severity_pass(task, loader, training=False):
    severity_model.train(training)
    head_name = "jsn_logits" if task == "JSN" else "erosion_logits"
    threshold_count = 4 if task == "JSN" else 5
    total_loss, count = 0.0, 0
    actual, predicted = [], []

    for images, grades in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        grades = grades.to(DEVICE, non_blocking=True)

        if training:
            severity_optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = severity_model(images)[head_name]
            targets = ordinal_targets(grades, threshold_count)
            loss = nn.functional.binary_cross_entropy_with_logits(logits, targets)
            if training:
                loss.backward()
                severity_optimizer.step()

        pred = (logits.detach() >= 0).sum(dim=1)
        total_loss += loss.item() * len(grades)
        count += len(grades)
        actual.extend(grades.cpu().tolist())
        predicted.extend(pred.cpu().tolist())

    return total_loss / max(count, 1), actual, predicted

In [32]:
# These are replaced in the training cell below with weighted training samplers.
# Keeping workers at zero prevents the Kaggle Python 3.13 shutdown warnings.
severity_loaders = {
    task: {
        split: DataLoader(
            SeverityDataset(severity_rows[task][split], augment=(split == "train")),
            batch_size=16,
            shuffle=(split == "train"),
            num_workers=0,
            pin_memory=False,
        )
        for split in ("train", "val", "test")
    }
    for task in ("JSN", "BE")
}
print("Severity loaders ready with num_workers=0.")


Severity loaders ready with num_workers=0.


In [33]:
from pathlib import Path
from shutil import copy2
from tqdm import tqdm
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import mean_absolute_error
import numpy as np
import torch

severity_path = CHECKPOINTS / "ra_severity.pt"
backup_path = CHECKPOINTS / "ra_severity_before_balancing.pt"

# Keep the checkpoint from your previous run.
if severity_path.is_file() and not backup_path.exists():
    copy2(severity_path, backup_path)
    print("Previous checkpoint backed up to:", backup_path)

# Rebuild loaders with num_workers=0 to avoid multiprocessing cleanup warnings.
# Square-root inverse weighting mildly increases the chance of seeing rare grades.
severity_loaders = {}

for task in ("JSN", "BE"):
    train_rows = severity_rows[task]["train"]
    max_grade = 4 if task == "JSN" else 5

    grade_counts = np.bincount(
        [int(grade) for _, grade in train_rows],
        minlength=max_grade + 1,
    )
    sample_weights = [
        1.0 / np.sqrt(max(grade_counts[int(grade)], 1))
        for _, grade in train_rows
    ]
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(train_rows),
        replacement=True,
    )

    severity_loaders[task] = {
        "train": DataLoader(
            SeverityDataset(train_rows, augment=True),
            batch_size=16,
            sampler=sampler,
            num_workers=0,
            pin_memory=False,
        ),
        "val": DataLoader(
            SeverityDataset(severity_rows[task]["val"]),
            batch_size=16,
            shuffle=False,
            num_workers=0,
            pin_memory=False,
        ),
        "test": DataLoader(
            SeverityDataset(severity_rows[task]["test"]),
            batch_size=16,
            shuffle=False,
            num_workers=0,
            pin_memory=False,
        ),
    }
    print(f"{task} training grade counts:", grade_counts.tolist())

# Start a fresh model and optimizer for this retraining run.
severity_model = OrdinalSeverityClassifier(pretrained=True).to(DEVICE)
severity_optimizer = torch.optim.AdamW(
    severity_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

def _severity_targets(grades, threshold_count):
    threshold_ids = torch.arange(threshold_count, device=grades.device)
    return (grades[:, None] > threshold_ids[None, :]).float()

def _severity_pass(task, loader, training=False):
    severity_model.train(training)
    head = "jsn_logits" if task == "JSN" else "erosion_logits"
    threshold_count = 4 if task == "JSN" else 5
    loss_sum, sample_count = 0.0, 0
    actual, predicted = [], []

    for images, grades in tqdm(
        loader, desc=f"{task} {'train' if training else 'validation'}"
    ):
        images = images.to(DEVICE, non_blocking=True)
        grades = grades.to(DEVICE, non_blocking=True)

        if training:
            severity_optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = severity_model(images)[head]
            targets = _severity_targets(grades, threshold_count)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logits, targets
            )
            if training:
                loss.backward()
                severity_optimizer.step()

        guesses = (logits.detach() >= 0).sum(dim=1)
        loss_sum += loss.item() * len(grades)
        sample_count += len(grades)
        actual.extend(grades.cpu().tolist())
        predicted.extend(guesses.cpu().tolist())

    return loss_sum / max(sample_count, 1), actual, predicted

def _macro_grade_mae(actual, predicted, max_grade):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    per_grade_errors = [
        np.abs(predicted[actual == grade] - grade).mean()
        for grade in range(max_grade + 1)
        if np.any(actual == grade)
    ]
    return float(np.mean(per_grade_errors))

best_validation_score = float("inf")
severity_epochs = 10

for epoch in range(1, severity_epochs + 1):
    train_losses, val_losses, val_grade_errors = [], [], []

    for task in ("JSN", "BE"):
        train_loss, train_true, train_pred = _severity_pass(
            task, severity_loaders[task]["train"], training=True
        )
        val_loss, val_true, val_pred = _severity_pass(
            task, severity_loaders[task]["val"]
        )

        max_grade = 4 if task == "JSN" else 5
        train_accuracy = float(np.mean(np.asarray(train_true) == np.asarray(train_pred)))
        val_accuracy = float(np.mean(np.asarray(val_true) == np.asarray(val_pred)))
        grade_error = _macro_grade_mae(val_true, val_pred, max_grade)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_grade_errors.append(grade_error)

        print(
            f"  {task}: train loss={train_loss:.4f}, train exact-grade accuracy={train_accuracy:.3f} | "
            f"val loss={val_loss:.4f}, val exact-grade accuracy={val_accuracy:.3f}, "
            f"macro-grade MAE={grade_error:.3f}"
        )

    mean_train_loss = float(np.mean(train_losses))
    mean_val_loss = float(np.mean(val_losses))
    mean_grade_error = float(np.mean(val_grade_errors))

    print(
        f"Epoch {epoch}/{severity_epochs} | "
        f"train loss {mean_train_loss:.4f} | "
        f"val loss {mean_val_loss:.4f} | "
        f"mean val macro-grade MAE {mean_grade_error:.3f}"
    )

    # Select by macro-grade MAE so grade 0 does not dominate checkpoint selection.
    if mean_grade_error < best_validation_score:
        best_validation_score = mean_grade_error
        torch.save(
            {
                "state_dict": severity_model.state_dict(),
                "epoch": epoch,
                "validation_macro_grade_mae": mean_grade_error,
                "image_size": 224,
            },
            severity_path,
        )
        print("  Saved best balanced severity checkpoint:", severity_path)

print("Training complete. Best checkpoint:", severity_path)

JSN training grade counts: [9162, 1808, 609, 164, 152]
BE training grade counts: [10045, 1869, 536, 73, 0, 165]


JSN validation: 100%|██████████| 132/132 [00:21<00:00,  6.08it/s]


  JSN: train loss=0.3081, train exact-grade accuracy=0.600 | val loss=0.1910, val exact-grade accuracy=0.783, macro-grade MAE=0.793


BE validation: 100%|██████████| 140/140 [00:28<00:00,  4.85it/s]


  BE: train loss=0.2668, train exact-grade accuracy=0.586 | val loss=0.1732, val exact-grade accuracy=0.751, macro-grade MAE=1.259
Epoch 1/10 | train loss 0.2875 | val loss 0.1821 | mean val macro-grade MAE 1.026
  Saved best balanced severity checkpoint: /kaggle/working/ra_project/checkpoints/ra_severity.pt


...

  BE: train loss=0.1503, train exact-grade accuracy=0.758 | val loss=0.2033, val exact-grade accuracy=0.706, macro-grade MAE=1.448
Epoch 9/10 | train loss 0.1603 | val loss 0.2021 | mean val macro-grade MAE 1.117


JSN validation: 100%|██████████| 132/132 [00:11<00:00, 11.35it/s]


  JSN: train loss=0.1658, train exact-grade accuracy=0.801 | val loss=0.1963, val exact-grade accuracy=0.806, macro-grade MAE=0.677


BE validation: 100%|██████████| 140/140 [00:12<00:00, 11.03it/s]

  BE: train loss=0.1440, train exact-grade accuracy=0.779 | val loss=0.2104, val exact-grade accuracy=0.755, macro-grade MAE=1.704
Epoch 10/10 | train loss 0.1549 | val loss 0.2033 | mean val macro-grade MAE 1.190
Training complete. Best checkpoint: /kaggle/working/ra_project/checkpoints/ra_severity.pt


In [37]:
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
)

if not severity_path.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found: {severity_path}. Run Cell 20 first."
    )

saved = torch.load(severity_path, map_location=DEVICE, weights_only=False)
severity_model.load_state_dict(saved["state_dict"])
severity_model.eval()

print("Checkpoint epoch:", saved["epoch"])
print("Checkpoint:", severity_path)

for task in ("JSN", "BE"):
    test_loss, actual, predicted = _severity_pass(
        task, severity_loaders[task]["test"]
    )
    max_grade = 4 if task == "JSN" else 5

    y_true = np.asarray(actual)
    y_pred = np.asarray(predicted)
    grades_present = np.unique(y_true)

    # Average recall over grades that occur in the true test labels.
    balanced_acc = np.mean([
        (y_pred[y_true == grade] == grade).mean()
        for grade in grades_present
    ])

    print(f"\n{task} test loss:", round(test_loss, 4))
    print("Exact-grade accuracy:", round(accuracy_score(y_true, y_pred), 3))
    print("Balanced accuracy (grades present in test):", round(balanced_acc, 3))
    print("Mean absolute grade error:", round(mean_absolute_error(y_true, y_pred), 3))
    print("Per-grade report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=list(range(max_grade + 1)),
        target_names=[f"Grade {g}" for g in range(max_grade + 1)],
        zero_division=0,
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=list(range(max_grade + 1))))

Checkpoint epoch: 1
Checkpoint: /kaggle/working/ra_project/checkpoints/ra_severity.pt


JSN validation: 100%|██████████| 251/251 [00:22<00:00, 11.20it/s]



JSN test loss: 0.2293
Exact-grade accuracy: 0.771
Balanced accuracy (grades present in test): 0.25
Mean absolute grade error: 0.305
Per-grade report:
              precision    recall  f1-score   support

     Grade 0       0.81      0.96      0.88      3122
     Grade 1       0.29      0.10      0.15       584
     Grade 2       0.21      0.07      0.10       192
     Grade 3       0.21      0.09      0.13        64
     Grade 4       1.00      0.02      0.05        43

    accuracy                           0.77      4005
   macro avg       0.51      0.25      0.26      4005
weighted avg       0.70      0.77      0.72      4005

Confusion matrix:
[[3007   94   20    1    0]
 [ 513   61    5    5    0]
 [ 137   38   13    4    0]
 [  41   10    7    6    0]
 [   5    9   16   12    1]]


BE validation: 100%|██████████| 267/267 [00:24<00:00, 11.04it/s]


BE test loss: 0.1988
Exact-grade accuracy: 0.721
Balanced accuracy (grades present in test): 0.281
Mean absolute grade error: 0.388
Per-grade report:
              precision    recall  f1-score   support

     Grade 0       0.75      0.97      0.85      3118
     Grade 1       0.17      0.03      0.05       717
     Grade 2       0.18      0.03      0.04       351
     Grade 3       0.17      0.12      0.14        25
     Grade 4       0.00      0.00      0.00         0
     Grade 5       0.80      0.26      0.40        61

    accuracy                           0.72      4272
   macro avg       0.35      0.23      0.25      4272
weighted avg       0.60      0.72      0.64      4272

Confusion matrix:
[[3033   49   23    6    5    2]
 [ 693   19    4    1    0    0]
 [ 304   31    9    2    3    2]
 [  10    8    4    3    0    0]
 [   0    0    0    0    0    0]
 [   9    3    9    6   18   16]]


In [38]:
for path in sorted(CHECKPOINTS.glob("*.pt")):
    print(path.name, f"{path.stat().st_size / (1024**2):.1f} MB")

print("YOLO best checkpoint exists:", YOLO_BEST.is_file())

ra_binary.pt 27.1 MB
ra_severity.pt 27.1 MB
YOLO best checkpoint exists: True


In [39]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
import shutil

export_dir = Path("/kaggle/working/ra_project_export")
export_dir.mkdir(parents=True, exist_ok=True)
files_to_save = {
    "checkpoints/ra_binary.pt": CHECKPOINTS / "ra_binary.pt",
    "checkpoints/ra_severity.pt": CHECKPOINTS / "ra_severity.pt",
    "yolo/best.pt": Path(YOLO_BEST),
    "yolo/data.yaml": Path(yaml_path),
}
optional_files = {
    "checkpoints/ra_severity_before_balancing.pt":
        CHECKPOINTS / "ra_severity_before_balancing.pt",
}
for archive_name, source in files_to_save.items():
    if not source.is_file():
        raise FileNotFoundError(f"Missing required file: {source}")
    destination = export_dir / archive_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print("Added required:", archive_name)
for archive_name, source in optional_files.items():
    if source.is_file():
        destination = export_dir / archive_name
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        print("Added optional:", archive_name)

zip_path = Path("/kaggle/working/ra_project_export.zip")
with ZipFile(zip_path, "w", ZIP_DEFLATED, compresslevel=1) as archive:
    for file in export_dir.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(export_dir))
print("Archive size (MB):", round(zip_path.stat().st_size / (1024**2), 1))
display(FileLink(str(zip_path)))


Added required: checkpoints/ra_binary.pt
Added required: checkpoints/ra_severity.pt
Added required: yolo/best.pt
Added required: yolo/data.yaml
Archive size (MB): 56.0


/kaggle/working/ra_project_export.zip